# Exercise 1 — Tabular Classification
### Rice Type Classifier

---

## The Problem

A rice processing company receives mixed batches of two rice varieties —
**Cammeo** and **Osmancik** — and needs to sort them automatically
using physical measurements taken by a camera system.

You are the ML engineer. Given a dataset of grain measurements,
build a classifier that identifies the rice type.

---

## The Data

```python
!pip install opendatasets --quiet
import opendatasets as od
od.download("https://www.kaggle.com/datasets/mssmartypants/rice-type-classification")
```

A CSV file with physical measurements of individual rice grains:
area, perimeter, axis lengths, eccentricity, convex area, and more.
The target column is `Class`.

⚠️ **One thing worth knowing:** the raw feature values are on very different
scales. A model trained on un-normalized data will struggle.
How you handle that is up to you.

---

## What You Need to Deliver

A working Colab notebook that contains:

1. **A trained PyTorch model** that classifies rice type from grain measurements
2. **A training report** — loss and accuracy curves over epochs for both
   training and validation sets
3. **A test accuracy score** — your final number on held-out data
4. **A live inference demo** — given a set of raw measurements,
   the model outputs the predicted rice type

Your model must be built in **PyTorch** (`nn.Module`, training loop, DataLoader).
No sklearn classifiers.

---

In [ ]:
!pip install opendatasets --quiet
import opendatasets as od
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import matplotlib.pyplot as plt
import os

# Download Data (requires Kaggle credentials: kaggle.json)
try:
    od.download("https://www.kaggle.com/datasets/mssmartypants/rice-type-classification")
except Exception as e:
    print("Could not automatically download. Ensure kaggle.json is present.")

# 1. Load Data
csv_file = None
for root, dirs, files in os.walk('.'):
    for f in files:
        if f.endswith('.csv') and 'rice' in f.lower():
            csv_file = os.path.join(root, f)
            break

if csv_file is None:
    print("Please ensure the dataset is downloaded before running the rest of the cells.")
else:
    df = pd.read_csv(csv_file)
    print("Data loaded successfully! Shape:", df.shape)
    
    # 2. Preprocess Data
    X = df.drop(columns=['Class']).values
    y = df['Class'].values
    
    # Encode target (Cammeo / Osmancik to 0 / 1)
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    
    # Train / Val / Test Split
    X_train_val, X_test, y_train_val, y_test = train_test_split(X, y_encoded, test_size=0.15, random_state=42, stratify=y_encoded)
    X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.15, random_state=42, stratify=y_train_val)
    
    # Standardize features (fit ONLY on training data to prevent data leakage)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)
    
    # Convert to PyTorch Tensors
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
    
    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
    
    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)
    
    # PyTorch Dataset
    class RiceDataset(Dataset):
        def __init__(self, X, y):
            self.X = X
            self.y = y
        def __len__(self):
            return len(self.y)
        def __getitem__(self, idx):
            return self.X[idx], self.y[idx]
            
    # PyTorch DataLoaders
    train_loader = DataLoader(RiceDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)
    val_loader = DataLoader(RiceDataset(X_val_t, y_val_t), batch_size=32, shuffle=False)
    test_loader = DataLoader(RiceDataset(X_test_t, y_test_t), batch_size=32, shuffle=False)
    print("DataLoaders created!")

In [ ]:
class RiceClassifier(nn.Module):
    def __init__(self, input_dim):
        super(RiceClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
            nn.Sigmoid() # Binary classification
        )
        
    def forward(self, x):
        return self.network(x)

if 'csv_file' in locals() and csv_file is not None:
    # Instantiate model, loss, and optimizer
    input_dim = X_train_t.shape[1]
    model = RiceClassifier(input_dim)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    print(model)

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=50):
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    
    for epoch in range(epochs):
        # --- Training Phase ---
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * X_batch.size(0)
            preds = (outputs >= 0.5).float()
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
            
        train_losses.append(running_loss / total)
        train_accs.append(correct / total)
        
        # --- Validation Phase ---
        model.eval()
        running_val_loss = 0.0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                
                running_val_loss += loss.item() * X_batch.size(0)
                preds = (outputs >= 0.5).float()
                val_correct += (preds == y_batch).sum().item()
                val_total += y_batch.size(0)
                
        val_losses.append(running_val_loss / val_total)
        val_accs.append(val_correct / val_total)
        
        if (epoch+1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_losses[-1]:.4f} Acc: {train_accs[-1]:.4f} | Val Loss: {val_losses[-1]:.4f} Acc: {val_accs[-1]:.4f}")
            
    return train_losses, val_losses, train_accs, val_accs

if 'csv_file' in locals() and csv_file is not None:
    train_losses, val_losses, train_accs, val_accs = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=50)

In [ ]:
if 'csv_file' in locals() and csv_file is not None:
    # Plot Training Report
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    ax1.plot(train_losses, label='Train Loss')
    ax1.plot(val_losses, label='Val Loss')
    ax1.set_title('Loss Curves')
    ax1.legend()
    
    ax2.plot(train_accs, label='Train Acc')
    ax2.plot(val_accs, label='Val Acc')
    ax2.set_title('Accuracy Curves')
    ax2.legend()
    
    plt.show()

    # Final Test Accuracy Score
    model.eval()
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            outputs = model(X_batch)
            preds = (outputs >= 0.5).float()
            test_correct += (preds == y_batch).sum().item()
            test_total += y_batch.size(0)
            
    print(f"Final Test Accuracy: {test_correct / test_total:.4f}")
    print("Target Accuracy: ≥ 0.92")

In [ ]:
if 'csv_file' in locals() and csv_file is not None:
    print("--- Live Inference Demo ---")
    
    # Pick a random sample from the dataframe to act as a raw input
    random_row = df.sample(1)
    raw_features = random_row.drop(columns=['Class']).values
    true_label = random_row['Class'].values[0]
    
    print("\nRaw Input Features:")
    for k, v in random_row.drop(columns=['Class']).iloc[0].to_dict().items():
        print(f"  {k}: {v}")
    print(f"\nTrue Label: {true_label}")
    
    # 1. Preprocess: Scale using our fitted scaler
    scaled_features = scaler.transform(raw_features)
    
    # 2. Convert to PyTorch Tensor
    tensor_features = torch.tensor(scaled_features, dtype=torch.float32)
    
    # 3. Model Inference
    model.eval()
    with torch.no_grad():
        output = model(tensor_features)
        pred_prob = output.item()
        pred_class_idx = int(pred_prob >= 0.5)
        pred_label = le.inverse_transform([pred_class_idx])[0]
        
    print(f"\nModel Prediction: {pred_label}")
    print(f"Confidence: {max(pred_prob, 1-pred_prob):.4f}")
    print(f"Result: {'✅ Correct' if pred_label == true_label else '❌ Incorrect'}")